# 01 · Social Signal 面板与 attention episode

这一步的目标只有一个：把 2012–2021 的 firm-day attention 矩阵读进来，按写死的规则切出 episode，然后**用眼睛看几十条曲线**，确认切出来的东西是“讨论爆发”，而不是数据噪声。

数据：`data/raw/social_signal_index.dta`（Cookson, Lu, Mullins & Niessner 2024 的正式发布版，不是 replication package 里的 stand-in）。

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.load_data import load_social_signal, wide_attention, coverage_table
from src.episodes import segment, label_sheet, plot_examples, describe_path

pd.set_option("display.width", 120)
AS_OF = "2021-12-31"
ROOT

## 1. 读进来

注意：源文件只包含“当天至少 10 条消息”的 firm-day。所以缺失不是随机缺失，而是“当天几乎没人讨论”。这一点决定了后面 episode 结束时把缺失当成 quiet 处理。

In [ ]:
df = load_social_signal(AS_OF)
Z = wide_attention(df, AS_OF)

print(f"rows          : {len(df):,}")
print(f"firms         : {df.permno.nunique():,}")
print(f"dates         : {df.date.nunique():,}  ({df.date.min().date()} -> {df.date.max().date()})")
print(f"grid          : {Z.shape[0]:,} x {Z.shape[1]:,}")
print(f"observed      : {Z.notna().to_numpy().sum():,} ({Z.notna().to_numpy().mean():.1%})")
df.head()

In [ ]:
cov = coverage_table(df, AS_OF)
display(cov["n_obs"].describe(percentiles=[.1, .25, .5, .75, .9]).round(0).to_frame().T)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
axes[0].hist(cov["n_obs"], bins=50, color="#1f4e79")
axes[0].set_title("Days with an index, per firm")
axes[0].set_xlabel("observed firm-days")

axes[1].hist(df["attention"], bins=200, color="#1f4e79")
axes[1].set_yscale("log")
axes[1].set_title("Attention index (z-scored PC)")
axes[1].set_xlabel("z")
axes[1].set_xlim(-1, 10)
fig.tight_layout()

## 2. 切 episode

规则写死在 `src/episodes.py` 顶部：

- 开始：`attention >= 3 x 过去 60 个交易日均值`
- 结束：回落到**开始时冻结的基线**的 1.5 倍以下，并连续 5 个交易日
- 至少要看到 3 个观测日

基线在 episode 开始时冻结，而不是每天重算。否则一次爆发会把它自己的参照系抬高，平台型会在五周左右被误判为结束。

In [ ]:
ep = segment(Z, as_of=AS_OF)
print(ep["status"].value_counts().to_string())
print(f"firms with >=1 episode: {ep.permno.nunique():,} / {Z.shape[1]:,}")

complete = ep[ep.status == "complete"]
print(f"\nepisodes: {len(ep):,}  (complete {len(complete):,})")
print("\nlength in days:")
print(complete["n_days"].describe(percentiles=[.1, .25, .5, .75, .9]).round(1).to_string())
print("\nper year:")
print(ep["start"].dt.year.value_counts().sort_index().to_string())
ep.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
complete["n_days"].clip(upper=60).hist(bins=40, ax=axes[0], color="#1f4e79")
axes[0].set_title("Episode length (days, clipped at 60)")
axes[0].set_xlabel("trading days")

complete["n_days"].groupby(complete["start"].dt.year).median().plot(
    ax=axes[1], marker="o", color="#c0392b"
)
axes[1].set_title("Median episode length by year")
axes[1].set_xlabel("")
fig.tight_layout()

## 3. 用眼睛看

每块面板：蓝线是 attention 指数，黄色区间是切出来的 episode，红竖线是起点。这一步是唯一能判断“切得对不对”的办法——数字看不出来。

In [ ]:
png = plot_examples(ep, Z, out_path=ROOT / "notebooks" / "figures" / "episodes_preview.png", n=30)
print(png)

try:
    from IPython.display import Image, display
    display(Image(filename=str(png)))
except Exception:
    pass  # not running under Jupyter


## 4. 人工标注表

`shape_auto` 只是提示，最终以 `shape` 列为准（`single` / `double` / `plateau`）。
标完存成 `data/interim/episodes_labeled.csv`，`src/models.py` 会直接读它。

In [ ]:
sheet = label_sheet(ep, Z)
print(sheet["shape_auto"].value_counts().to_string())
print(f"\n{len(sheet):,} completed episodes to label")

out = ROOT / "data" / "interim" / "episodes_to_label.csv"
out.parent.mkdir(parents=True, exist_ok=True)
sheet.drop(columns=["path"]).to_csv(out, index=False)
print(f"wrote {out}")
sheet.drop(columns=["path"]).head(12)